In [5]:
import json
import joblib
import os
import shutil
from pathlib import Path
from azureml.core import Workspace, Model

# Get Azure ML workspace
ws = Workspace.from_config()

# Debug: Show current working directory
print(f"📍 Current Working Directory: {os.getcwd()}")

# Define paths - use absolute path 
# Outputs are in: /home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure/outputs/
notebook_dir = Path("/home/azureuser/cloudfiles/code/Users/Mahesh113274/MLOps_Azure")
outputs_dir = notebook_dir / "outputs"
model_path = outputs_dir / "model.pkl"
metrics_json_path = outputs_dir / "metrics.json"

print(f"Loading from output directory: {outputs_dir}")

# Load the trained model
model = joblib.load(model_path)
print("✅ Model loaded successfully")

# Load evaluation metrics
with open(metrics_json_path, "r") as f:
    metrics = json.load(f)
print("✅ Metrics loaded successfully")

# Define test thresholds
test_thresholds = {
    'accuracy': 0.60,
    'precision': 0.60,
    'recall': 0.60,
    'f1_score': 0.60
}

# Function to check if metrics pass thresholds
def tests_pass(metrics, thresholds):
    for metric, threshold in thresholds.items():
        value = metrics.get(metric)
        if value is None:
            print(f"⚠️ Metric '{metric}' not found in metrics.json")
            return False
        if value < threshold:
            print(f"❌ Test failed: {metric} = {value:.4f} < threshold {threshold}")
            return False
    return True

# Run tests before registering model
if tests_pass(metrics, test_thresholds):
    print("✅ All tests passed. Proceeding with model registration...")

    # Copy model to current working directory (required by Azure ML)
    current_cwd = Path(os.getcwd())
    local_model_path = current_cwd / "model.pkl"
    shutil.copy(str(model_path), str(local_model_path))
    print(f"✅ Model copied to: {local_model_path}")

    # Register model in Azure ML
    model_name = "CreditCard_Fraud_Detection"

    registered_model = Model.register(
        workspace=ws,
        model_path=str(local_model_path),
        model_name=model_name,
        tags={
            "role": "challenger",
            "status": "staging",
            "framework": "sklearn",
            "accuracy": str(metrics.get('accuracy', 0)),
            "precision": str(metrics.get('precision', 0)),
            "recall": str(metrics.get('recall', 0))
        },
        description="Credit Card Fraud Detection Model trained with Random Forest"
    )

    print(f"\n✅ Model registered in Azure ML")
    print(f"   Model Name: {registered_model.name}")
    print(f"   Model Version: {registered_model.version}")
    print(f"   Model ID: {registered_model.id}")

    # Display model details
    print(f"\n📊 Model Metrics:")
    for metric_name, value in metrics.items():
        print(f"   {metric_name}: {value:.4f}")

    # Cleanup
    local_model_path.unlink()
    print("\n✅ Cleanup completed")

else:
    print("❌ Model failed the evaluation tests and will NOT be registered.")

Using output directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/mahesh113274ds11v2/code/Users/Mahesh113274/MLOps_Azure/outputs
✅ All tests passed. Proceeding with model registration...
Registering model CreditCard_Fraud_Detection

✅ Model registered in Azure ML
   Model Name: CreditCard_Fraud_Detection
   Model Version: 1
   Model ID: CreditCard_Fraud_Detection:1

📊 Model Metrics:
   accuracy: 0.9994
   precision: 1.0000
   recall: 0.7463
   f1_score: 0.8547
